# Optimasi Parameter Holt-Winters (Triple Exponential Smoothing)
Notebook ini digunakan untuk mencari parameter terbaik (Alpha, Beta, Gamma) untuk setiap bandara domestik dan internasional menggunakan metode Grid Search.

**Modifikasi:**
- Menghapus data periode Covid-19 (2020-2022).
- Menampilkan 5 kombinasi parameter terbaik berdasarkan nilai MAPE terendah.

In [8]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

file_path = 'target_penumpang_domestik_internasional_2006_2024.xlsx'
df_raw = pd.read_excel(file_path)

bulan_map = {
    'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'Mei':5, 'Jun':6,
    'Jul':7, 'Agu':8, 'Sep':9, 'Okt':10, 'Nov':11, 'Des':12
}
df_raw['Bulan_num'] = df_raw['Bulan'].map(bulan_map)
df_raw['Tanggal'] = pd.to_datetime(df_raw['Tahun'].astype(str) + '-' + df_raw['Bulan_num'].astype(str) + '-01')
df_raw = df_raw.sort_values('Tanggal').reset_index(drop=True)

df = df_raw[~df_raw['Tahun'].isin([2020, 2021, 2022])].reset_index(drop=True)

print(f"Data Full (Tanpa Covid) loaded. Range: {df['Tanggal'].min()} to {df['Tanggal'].max()}")

Data Full (Tanpa Covid) loaded. Range: 2006-01-01 00:00:00 to 2025-12-01 00:00:00


## Bagian 1: Optimasi Menggunakan Seluruh Data (Tahun 2006 - 2024, Tanpa Covid)

In [10]:
bandara_cols = [
    'soekarno_hatta_jakarta_domestik', 'ngurah_rai_bali_domestik', 
    'juanda_surabaya_domestik', 'kualanamu_medan_domestik', 
    'hasanudin_makassar_domestik', 'soekarno_hatta_jakarta_internasional', 
    'ngurah_rai_bali_internasional', 'juanda_surabaya_internasional', 
    'kualanamu_medan_internasional'
]

values = np.arange(0.01, 0.81, 0.05)
seasonal_period = 12

for col in bandara_cols:
    print(f"\nANALYZING: {col.upper()} (FULL DATA)")
    data = df[col].values
    split_idx = int(len(data) * 0.8)
    train, test = data[:split_idx], data[split_idx:]
    
    all_results = []
    for a in values:
        for b in values:
            for g in values:
                try:
                    model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=seasonal_period)
                    fit = model.fit(smoothing_level=a, smoothing_trend=b, smoothing_seasonal=g, optimized=False)
                    forecast = fit.forecast(len(test))
                    r2 = r2_score(test, forecast)
                    mape = np.mean(np.abs((test - forecast) / test)) * 100
                    all_results.append({'Alpha': round(a, 2), 'Beta': round(b, 2), 'Gamma': round(g, 2), 'R2': round(r2, 4), 'MAPE (%)': round(mape, 2)})
                except: continue
    
    top_5 = pd.DataFrame(all_results).sort_values('MAPE (%)', ascending=True).head(5)
    display(top_5)


ANALYZING: SOEKARNO_HATTA_JAKARTA_DOMESTIK (FULL DATA)


,Alpha,Beta,Gamma,R2,MAPE (%)
530,0.11,0.06,0.11,0.6773,5.20
531,0.11,0.06,0.16,0.6682,5.28
529,0.11,0.06,0.06,0.6553,5.31
294,0.06,0.11,0.31,0.6720,5.42
293,0.06,0.11,0.26,0.6662,5.51



ANALYZING: NGURAH_RAI_BALI_DOMESTIK (FULL DATA)


,Alpha,Beta,Gamma,R2,MAPE (%)
308,0.06,0.16,0.21,0.6506,6.51
788,0.16,0.06,0.21,0.6569,6.56
307,0.06,0.16,0.16,0.6207,6.64
309,0.06,0.16,0.26,0.6539,6.64
787,0.16,0.06,0.16,0.6521,6.67



ANALYZING: JUANDA_SURABAYA_DOMESTIK (FULL DATA)


...

,Alpha,Beta,Gamma,R2,MAPE (%)
1437,0.26,0.46,0.66,0.1663,16.10
706,0.11,0.61,0.11,0.1336,16.64
721,0.11,0.66,0.06,0.0571,16.91
736,0.11,0.71,0.01,0.0672,16.95
692,0.11,0.56,0.21,0.0490,17.12



ANALYZING: JUANDA_SURABAYA_INTERNASIONAL (FULL DATA)


,Alpha,Beta,Gamma,R2,MAPE (%)
66,0.01,0.21,0.11,0.1628,9.05
65,0.01,0.21,0.06,0.1972,9.05
82,0.01,0.26,0.11,0.1583,9.33
67,0.01,0.21,0.16,0.0630,9.46
64,0.01,0.21,0.01,0.1314,9.51



ANALYZING: KUALANAMU_MEDAN_INTERNASIONAL (FULL DATA)


,Alpha,Beta,Gamma,R2,MAPE (%)
97,0.01,0.31,0.06,0.4106,8.09
96,0.01,0.31,0.01,0.2283,8.39
82,0.01,0.26,0.11,0.3631,8.53
83,0.01,0.26,0.16,0.3496,8.59
67,0.01,0.21,0.16,0.3028,8.78


## Bagian 2: Optimasi Menggunakan Data Mulai Tahun 2015 (Tanpa Covid)
Data yang digunakan adalah data tahun 2015 - 2019 dan 2023 - 2024.

In [9]:
df_2015 = df[df['Tahun'] >= 2015].reset_index(drop=True)

print(f"Data mulai 2015 (Tanpa Covid) loaded. Range: {df_2015['Tanggal'].min()} to {df_2015['Tanggal'].max()}")
print(f"Jumlah baris data: {len(df_2015)}")

for col in bandara_cols:
    print(f"\nANALYZING: {col.upper()} (MULAI TAHUN 2015)")
    data = df_2015[col].values
    split_idx = int(len(data) * 0.8)
    train, test = data[:split_idx], data[split_idx:]
    
    all_results_2015 = []
    for a in values:
        for b in values:
            for g in values:
                try:
                    model = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=seasonal_period)
                    fit = model.fit(smoothing_level=a, smoothing_trend=b, smoothing_seasonal=g, optimized=False)
                    forecast = fit.forecast(len(test))
                    r2 = r2_score(test, forecast)
                    mape = np.mean(np.abs((test - forecast) / test)) * 100
                    all_results_2015.append({'Alpha': round(a, 2), 'Beta': round(b, 2), 'Gamma': round(g, 2), 'R2': round(r2, 4), 'MAPE (%)': round(mape, 2)})
                except: continue
    
    top_5_2015 = pd.DataFrame(all_results_2015).sort_values('MAPE (%)', ascending=True).head(5)
    display(top_5_2015)

Data mulai 2015 (Tanpa Covid) loaded. Range: 2015-01-01 00:00:00 to 2025-12-01 00:00:00
Jumlah baris data: 96

ANALYZING: SOEKARNO_HATTA_JAKARTA_DOMESTIK (MULAI TAHUN 2015)


,Alpha,Beta,Gamma,R2,MAPE (%)
470,0.06,0.66,0.31,0.7581,4.33
484,0.06,0.71,0.21,0.7588,4.36
87,0.01,0.26,0.36,0.7652,4.38
86,0.01,0.26,0.31,0.7602,4.38
456,0.06,0.61,0.41,0.7442,4.40



ANALYZING: NGURAH_RAI_BALI_DOMESTIK (MULAI TAHUN 2015)


,Alpha,Beta,Gamma,R2,MAPE (%)
3906,0.76,0.21,0.11,0.6914,5.37
2610,0.51,0.16,0.11,0.7098,5.38
3634,0.71,0.16,0.11,0.6982,5.39
3378,0.66,0.16,0.11,0.6920,5.43
2353,0.46,0.16,0.06,0.6881,5.44



ANALYZING: JUANDA_SURABAYA_DOMESTIK (MULAI TAHUN 2015)


...

,Alpha,Beta,Gamma,R2,MAPE (%)
2297,0.41,0.76,0.46,0.4637,16.26
2296,0.41,0.76,0.41,0.4081,16.46
2295,0.41,0.76,0.36,0.3623,16.87
2294,0.41,0.76,0.31,0.3432,16.95
2298,0.41,0.76,0.51,0.4955,17.37



ANALYZING: JUANDA_SURABAYA_INTERNASIONAL (MULAI TAHUN 2015)


,Alpha,Beta,Gamma,R2,MAPE (%)
3602,0.71,0.06,0.11,0.0451,7.95
3874,0.76,0.11,0.11,0.0412,7.96
3330,0.66,0.01,0.11,0.0384,7.98
1721,0.31,0.56,0.46,0.0691,7.98
3074,0.61,0.01,0.11,0.0397,7.98



ANALYZING: KUALANAMU_MEDAN_INTERNASIONAL (MULAI TAHUN 2015)


,Alpha,Beta,Gamma,R2,MAPE (%)
1551,0.31,0.01,0.76,0.5881,4.67
1295,0.26,0.01,0.76,0.5310,4.71
1550,0.31,0.01,0.71,0.5625,4.72
1549,0.31,0.01,0.66,0.5305,4.77
1807,0.36,0.01,0.76,0.5860,4.78
